# AERIS — data access and first inspection

**EGN 6216, Module 7.** This notebook retrieves the LBNL-2019-2 target CSVs from OEDI and G36-1wk source runs from Figshare, caches the archives locally, and inspects actual CSV data. It is an exploratory notebook, not a training or deployment pipeline.

**Project scope:** G36-1wk is the simulated source. LBNL-2019-2 is a separate real FLEXLAB target planned for the later cross-setting study. Any comparison of these domains must first check seasons, sensors, units and scenario labels. The source files alone do not validate target coverage.

**Provenance:** Source: Ghalamsiah et al., *Labeled Datasets for Air Handling Units Operating in Faulted and Fault-free States*, [Figshare DOI 10.6084/m9.figshare.29297999](https://doi.org/10.6084/m9.figshare.29297999), dataset license **CC BY 4.0** per Figshare. Target: Lin and Mitchell (2019), *Data Sets for Evaluation of Building Fault Detection and Diagnostics Algorithms*, [OEDI DOI 10.25984/1824861](https://doi.org/10.25984/1824861), license **CC BY 4.0** per the U.S. data catalog. The later AHU comparison paper calls the target SZVAV/SZCAV files “LBNL-2019-2.” The published articles have separate terms. Cells below record UTC retrieval dates, file names and SHA-256 digests. No pretrained models are used.

Run from top to bottom with an internet connection. The initial archive download may take several minutes. The cache is under `data/` and is ignored by Git. **Save this notebook with outputs before committing.**


In [16]:
# Standard-library network and archive tools keep dataset access credential-free.
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import io
import platform
import re
import urllib.request
import zipfile

import pandas as pd
from IPython.display import display

ARTICLE_ID = 29297999
ARTICLE_API = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Copy an exact file name from the next cell only if the automatic choice fails.
DATA_FILE_NAME = None
# If needed, copy exact CSV paths from the archive inventory below.
DATA_CSV_NAMES = [
    "Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/BaselineSystem.csv",
    "Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/OADamStuck_65.csv",
]

retrieved_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
print(f"Retrieval date (UTC): {retrieved_at}")
print(f"Python: {platform.python_version()} | pandas: {pd.__version__}")
print(f"Cache directory: {DATA_DIR.resolve()}")


Retrieval date (UTC): 2026-10-06T02:19:16+00:00
Python: 3.11.9 | pandas: 2.2.3
Cache directory: C:\MSAIS\EGN6216-AI SYSTEMS\AHU_PROJECT\aeris-data-warmup_updated\data


## 1. Real target setting: LBNL-2019-2 (FLEXLAB)

The 2019 OEDI archive includes several systems. Only its **SZVAV.csv** and **SZCAV.csv** correspond to the real test-facility AHU target discussed in the project plan. The 2019 dataset release is publicly accessible under CC BY 4.0; its inventory PDF documents channels, units and fault scenarios. This notebook accesses the original ZIP directly, caches it under `data/`, and reads those two CSV members without extracting the whole archive.


In [17]:
# Fetch the 2019 LBNL/OEDI archive directly from its publisher.
# The release has no access token and is small enough for a quick first check.
TARGET_DOI = "10.25984/1824861"
TARGET_RECORD = "https://data.openei.org/submissions/910"
TARGET_URL = (
    "https://data.openei.org/files/910/"
    "Data%20Sets%20for%20AFDD%20Evauluation%20of%20Building%20FDD%20Algorithms.zip"
)
TARGET_VERSION = "OEDI release last updated 2021-10-15"
TARGET_LICENSE = "CC BY 4.0"
target_retrieved_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
target_zip = DATA_DIR / "lbnl_2019_afdd.zip"
if not target_zip.exists():
    request = urllib.request.Request(TARGET_URL, headers={"User-Agent": "AERIS-course-data-audit/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response, target_zip.open("wb") as out:
        while block := response.read(1024 * 1024):
            out.write(block)
else:
    print("Using cached OEDI archive.")
if not zipfile.is_zipfile(target_zip):
    raise ValueError("OEDI response is not a ZIP; delete the cached file and retry.")
target_sha256 = hashlib.sha256(target_zip.read_bytes()).hexdigest()
print(f"Target: {TARGET_RECORD} | {TARGET_DOI} | {TARGET_VERSION} | {TARGET_LICENSE}")
print(f"Retrieved UTC: {target_retrieved_at}")
print(f"Cached ZIP: {target_zip}, {target_zip.stat().st_size / 1e6:.2f} MB")
print(f"SHA-256: {target_sha256}")


Using cached OEDI archive.
Target: https://data.openei.org/submissions/910 | 10.25984/1824861 | OEDI release last updated 2021-10-15 | CC BY 4.0
Retrieved UTC: 2026-10-06T02:19:16+00:00
Cached ZIP: data\lbnl_2019_afdd.zip, 7.01 MB
SHA-256: 68c58568e3f0e4e7d0f0de3194a2ce7ca80703c057325ee402a94d3eab5a76c1


In [18]:
# The two real AHU modes are separate target settings; inspect both.
target_archive = zipfile.ZipFile(target_zip)
target_csv_names = [name for name in target_archive.namelist() if name.lower().endswith(".csv")]
display(pd.DataFrame({"OEDI_CSV_member": target_csv_names}))
selected_target_names = [
    name for wanted in ("szvav.csv", "szcav.csv")
    for name in target_csv_names if Path(name).name.lower() == wanted
]
if len(selected_target_names) != 2:
    raise ValueError("Expected SZVAV.csv and SZCAV.csv; inspect the ZIP member names above.")
target_frames = {}
for name in selected_target_names:
    with target_archive.open(name) as stream:
        target_frames[name] = pd.read_csv(stream, low_memory=False)
    print(f"{name}: {len(target_frames[name]):,} rows × {len(target_frames[name].columns)} fields")


,OEDI_CSV_member
0,RTU.csv
1,SZCAV.csv
2,SZVAV.csv
3,MZVAV-1.csv
4,MZVAV-2-1.csv
5,MZVAV-2-2.csv


SZVAV.csv: 15,839 rows × 16 fields
SZCAV.csv: 21,600 rows × 16 fields


In [19]:
# Display shape, all data types, five rows and basic data quality for
# each real target setting. The fault-ground-truth column, if present, is
# described in the publisher's inventory; do not equate it with fault type.
target_quality = []
for name, frame in target_frames.items():
    print(f"\nTarget CSV: {name}")
    display(frame.dtypes.rename("pandas_dtype").to_frame())
    display(frame.head(5))
    target_quality.append({
        "csv": name, "rows": len(frame), "fields": len(frame.columns),
        "missing_cells": int(frame.isna().sum().sum()),
        "columns_with_missing": int((frame.isna().sum() > 0).sum()),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
    })
    print("Columns with missing values (up to 15):")
    missing = frame.isna().sum().sort_values(ascending=False)
    display(missing[missing > 0].head(15).to_frame("missing"))
    print("Numeric ranges (up to 20; check units in publisher inventory):")
    display(frame.select_dtypes(include="number").agg(["min", "max"]).T.head(20))
    potential_labels = [column for column in frame.columns if any(
        word in column.lower() for word in ("ground truth", "fault detection", "fault type", "label")
    )]
    for column in potential_labels[:3]:
        print(f"Value counts for candidate label field {column!r} (including missing):")
        display(frame[column].value_counts(dropna=False).head(20).to_frame("rows"))

display(pd.DataFrame(target_quality))



Target CSV: SZVAV.csv


,pandas_dtype
Datetime,object
AHU: Supply Air Temperature,float64
AHU: Supply Air Temperature Heating Set Point,float64
AHU: Supply Air Temperature Cooling Set Point,float64
AHU: Outdoor Air Temperature,object
AHU: Mixed Air Temperature,float64
AHU: Return Air Temperature,float64
AHU: Supply Air Fan Status,int64
AHU: Supply Air Fan Speed Control Signal,float64
AHU: Outdoor Air Damper Control Signal,float64


,Datetime,AHU: Supply Air Temperature,AHU: Supply Air Temperature Heating Set Point,AHU: Supply Air Temperature Cooling Set Point,AHU: Outdoor Air Temperature,AHU: Mixed Air Temperature,AHU: Return Air Temperature,AHU: Supply Air Fan Status,AHU: Supply Air Fan Speed Control Signal,AHU: Outdoor Air Damper Control Signal,AHU: Return Air Damper Control Signal,AHU: Exhaust Air Damper Control Signal,AHU: Cooling Coil Valve Control Signal,AHU: Heating Coil Valve Control Signal,Occupancy Mode Indicator,Fault Detection Ground Truth
0,9/11/2017 0:00,72.29,NaN,NaN,81.11,78.56,76.38,1,0.1,0.0,1.0,-0.04,0.0,0.23,0,1
1,9/11/2017 0:01,72.30,NaN,NaN,81.11,78.56,76.39,1,0.1,0.0,1.0,-0.04,0.0,0.23,0,1
2,9/11/2017 0:02,72.37,NaN,NaN,81.09,78.56,76.38,1,0.1,0.0,1.0,-0.04,0.0,0.22,0,1
3,9/11/2017 0:03,72.29,NaN,NaN,81.06,78.51,76.38,1,0.1,0.0,1.0,-0.04,0.0,0.22,0,1
4,9/11/2017 0:04,71.82,NaN,NaN,81.05,78.41,76.37,1,0.1,0.0,1.0,-0.04,0.0,0.24,0,1


Columns with missing values (up to 15):


,missing
AHU: Supply Air Temperature Cooling Set Point,590
AHU: Supply Air Temperature Heating Set Point,590


...

Numeric ranges (up to 20; check units in publisher inventory):


,min,max
AHU: Supply Air Temperature,49.21,88.83
AHU: Supply Air Temperature Heating Set Point,50.00,86.00
AHU: Supply Air Temperature Cooling Set Point,50.00,86.00
AHU: Outdoor Air Temperature,44.60,71.18
AHU: Return Air Temperature,58.45,83.23
AHU: Supply Air Fan Status,0.00,1.00
AHU: Supply Air Fan Speed Control Signal,0.00,0.50
AHU: Outdoor Air Damper Control Signal,0.00,1.00
AHU: Return Air Damper Control Signal,0.00,1.00
AHU: Exhaust Air Damper Control Signal,0.00,0.50


Value counts for candidate label field 'Fault Detection Ground Truth' (including missing):


,rows
Fault Detection Ground Truth,
1,20160
0,1440


,csv,rows,fields,missing_cells,columns_with_missing,exact_duplicate_rows
0,SZVAV.csv,15839,16,1180,2,0
1,SZCAV.csv,21600,16,0,0,0


## 2. Simulated source setting: G36-1wk

The publisher's Figshare record lists the source files. A selected archive is downloaded to the ignored `data/` cache, and one or two CSV scenario runs are inspected. This is **not** a full inventory or a claim that the source and target features are aligned.


In [20]:
# Fetch *current* source metadata; do not hard-code a file ID or download URL.
import json

request = urllib.request.Request(ARTICLE_API, headers={"User-Agent": "AERIS-course-data-audit/1.0"})
with urllib.request.urlopen(request, timeout=30) as response:
    article = json.load(response)

publisher_files = article.get("files", [])
print(f"Source: {article.get('title')}")
print(f"DOI: {article.get('doi')} | version: {article.get('version')}")
print(f"License in Figshare metadata: {article.get('license')}")
print(f"Publisher files listed: {len(publisher_files)}")
files_table = pd.DataFrame([
    {"name": f["name"], "size_MB": round(f.get("size", 0) / 1e6, 1),
     "file_id": f.get("id"), "has_download_url": bool(f.get("download_url"))}
    for f in publisher_files
])
display(files_table)
if not publisher_files:
    raise ValueError("Figshare did not list downloadable files. Verify article ID and network access.")


Source: Labeled Datasets for Air Handling Units Operating in Faulted and Fault-free States
DOI: 10.6084/m9.figshare.29297999.v3 | version: 3
License in Figshare metadata: {'value': 1, 'name': 'CC BY 4.0', 'url': 'https://creativecommons.org/licenses/by/4.0/'}
Publisher files listed: 1


,name,size_MB,file_id,has_download_url
0,Public_ScientificData_AHUFaults.zip,928.7,57268553,True


In [21]:
# Prefer an explicitly named G36-1wk archive. If there is one general
# archive, it can be inspected too; otherwise choose its exact name above.
if DATA_FILE_NAME:
    matches = [f for f in publisher_files if f["name"] == DATA_FILE_NAME]
else:
    matches = [f for f in publisher_files if "g36-1wk" in f["name"].lower()]
    if not matches:
        archives = [f for f in publisher_files if f["name"].lower().endswith(".zip")]
        matches = archives if len(archives) == 1 else []

if len(matches) != 1:
    raise ValueError(
        f"Could not choose one source file (matches: {[f['name'] for f in matches]}). "
        "Inspect files_table and set DATA_FILE_NAME to the G36-1wk archive's exact name."
    )
source_file = matches[0]
print("Selected publisher file:", source_file["name"])
print("Download size (MB):", round(source_file.get("size", 0) / 1e6, 1))


Selected publisher file: Public_ScientificData_AHUFaults.zip
Download size (MB): 928.7


In [22]:
# Download only the chosen source file, with a local cache for reruns.
# The publisher supplies an anonymous HTTPS URL, so no token is stored.
local_archive = DATA_DIR / Path(source_file["name"]).name
expected_bytes = source_file.get("size")
if local_archive.exists() and expected_bytes and local_archive.stat().st_size != expected_bytes:
    print("Cached size differs from publisher metadata; fetching again.")
    local_archive.unlink()

if not local_archive.exists():
    if not source_file.get("download_url"):
        raise ValueError("Selected Figshare file has no direct download URL.")
    temporary_file = local_archive.with_name(local_archive.name + ".part")
    try:
        request = urllib.request.Request(
            source_file["download_url"],
            headers={"User-Agent": "AERIS-course-data-audit/1.0"},
        )
        with urllib.request.urlopen(request, timeout=120) as response, temporary_file.open("wb") as out:
            transferred = 0
            last_report = 0
            while block := response.read(1024 * 1024):
                out.write(block)
                transferred += len(block)
                if transferred - last_report >= 100 * 1024 * 1024:
                    print(f"Downloaded {transferred / 1e6:.0f} MB...")
                    last_report = transferred
        temporary_file.replace(local_archive)
    finally:
        temporary_file.unlink(missing_ok=True)
else:
    print("Using already downloaded local cache.")

digest = hashlib.sha256()
with local_archive.open("rb") as stream:
    for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
        digest.update(block)
print(f"Local file: {local_archive} ({local_archive.stat().st_size / 1e6:.1f} MB)")
print(f"SHA-256: {digest.hexdigest()}")


Using already downloaded local cache.
Local file: data\Public_ScientificData_AHUFaults.zip (928.7 MB)
SHA-256: 243f48c505dc7320cfef8d22911d7fa82fa6f9a19e99132d29aa1d988ebff954


In [23]:
# Inspect member names before assuming that the archive contains
# the planned AHU/season/fault scenarios. Keep the archive, not extracted copies.
if zipfile.is_zipfile(local_archive):
    archive = zipfile.ZipFile(local_archive)
    all_csv_names = [name for name in archive.namelist() if name.lower().endswith(".csv")]
    if not all_csv_names:
        nested_archives = [name for name in archive.namelist() if name.lower().endswith(".zip")]
        preferred_nested = [name for name in nested_archives if "g36-1wk" in name.lower()]
        nested_selection = preferred_nested or (nested_archives if len(nested_archives) == 1 else [])
        if len(nested_selection) == 1:
            print("Opening nested archive:", nested_selection[0])
            # An in-memory copy avoids writing many large intermediate files.
            archive = zipfile.ZipFile(io.BytesIO(archive.read(nested_selection[0])))
            all_csv_names = [name for name in archive.namelist() if name.lower().endswith(".csv")]
    g36_names = [name for name in all_csv_names if "g36-1wk" in name.lower()]
    candidate_names = g36_names or all_csv_names
    print(f"CSV members: {len(all_csv_names)} | G36-1wk paths: {len(g36_names)}")
    print("First 25 CSV paths:")
    display(pd.DataFrame({"csv_path": candidate_names[:25]}))
    if not candidate_names:
        print("First 25 archive members:")
        display(pd.DataFrame({"member": archive.namelist()[:25]}))
        raise ValueError("No direct CSV found. Inspect archive members for nested archives.")
elif local_archive.suffix.lower() == ".csv":
    archive = None
    candidate_names = [local_archive.name]
    print("Selected file is a direct CSV.")
else:
    raise ValueError(f"Unsupported published file type: {local_archive.suffix}. Pick a ZIP or CSV.")


CSV members: 686 | G36-1wk paths: 362
First 25 CSV paths:


,csv_path
0,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...
1,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...
2,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...
3,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...
4,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...
5,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...
6,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...
7,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...
8,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...
9,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...


In [24]:
matches = [
    name for name in candidate_names
    if "/coolingseason/" in name.lower()
    and (
        ("coi" in name.rsplit("/", 1)[-1].lower() and "val" in name.rsplit("/", 1)[-1].lower())
        or "dam" in name.rsplit("/", 1)[-1].lower()
    )
]
print(f"Relevant-looking cooling-season files: {len(matches)}")
for name in matches[:80]:
    print(name)

Relevant-looking cooling-season files: 16
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/ConVAVSoukDam_05.csv
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/ConVAVSoukDam_5.csv
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/CooCoiValLea_001.csv
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/CooCoiValLea_01.csv
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/CooCoiValStuck_0.csv
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/CooCoiValStuck_100.csv
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/CooCoiValStuck_15.csv
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/CooCoiValStuck_5.csv
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/CooCoiValStuck_65.csv
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/OADamLea_001.csv
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/OADamLea_01.csv
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/OADamStuck_0.csv
Public_ScientificData_AHUFaults/0

In [25]:
# Select one normal and one damper/valve scenario if their *filenames*
# identify them. Otherwise use two illustrative CSVs and leave their labels unknown.
if DATA_CSV_NAMES:
    selected_names = DATA_CSV_NAMES
else:
    baseline = [n for n in candidate_names if re.search(
        r"baseline|normal|fault[-_ ]?free|no[-_ ]?fault", n, re.I)]
    fault = [n for n in candidate_names if re.search(r"damper|valve|dam|val", n, re.I)]
    if baseline and fault:
        selected_names = [baseline[0], next((n for n in fault if n != baseline[0]), fault[0])]
    else:
        selected_names = candidate_names[:2]

if len(selected_names) < 1 or any(n not in candidate_names for n in selected_names):
    raise ValueError("Select one or two exact CSV paths from the inventory above.")
if len(selected_names) == 1:
    print("Only one CSV was available; class balance cannot be assessed.")

frames = {}
for name in selected_names:
    # Open one member at a time instead of extracting a multi-gigabyte dataset.
    if archive is None:
        frame = pd.read_csv(local_archive, low_memory=False)
    else:
        with archive.open(name) as stream:
            frame = pd.read_csv(stream, low_memory=False)
    frames[name] = frame
    print(f"Loaded {name}: {len(frame):,} rows × {len(frame.columns)} fields")

display(pd.DataFrame({"selected_csv_path": selected_names}))


Loaded Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/BaselineSystem.csv: 10,081 rows × 114 fields
Loaded Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/OADamStuck_65.csv: 10,081 rows × 114 fields


,selected_csv_path
0,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...
1,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...


In [26]:
# Report every field's dtype and show the first five *real* rows.
# Some files use abbreviations; the publisher's explanations PDF is needed
# before assigning physical units or mapping columns across source/target.
for name, frame in frames.items():
    print(f"\n{name}")
    print(f"Rows: {len(frame):,}; fields: {len(frame.columns)}")
    display(frame.dtypes.rename("pandas_dtype").to_frame())
    display(frame.head(5))



Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/BaselineSystem.csv
Rows: 10,081; fields: 114


,pandas_dtype
time (s),int64
East Zone Room Temperature (°K),float64
East Zone Room Cooling Setpoint (°K),float64
East Zone Room Heating Setpoint (°K),float64
East Zone Reheating Coil Valve Power Control Signal (W),float64
...,...
"Speed Control Signal of the Cooling Water Pump (fraction, 0: pump speed should be 0% to 1: pump speed should be 100%)",float64
"Speed Control Signal of the Hot Water Pump (fraction, 0: pump speed should be 0% to 1: pump speed should be 100%)",float64
"Cooling Mode Control Signal (1: free cooling , 2: partial mechanical, 3: full mechanical, 4: off)",float64
"Chiller Plant Equipment Operation Signal (0: off, 1: on)",float64


,time (s),East Zone Room Temperature (°K),East Zone Room Cooling Setpoint (°K),East Zone Room Heating Setpoint (°K),East Zone Reheating Coil Valve Power Control Signal (W),East Zone Reheating Coil Valve Power Consumption (W),East Zone VAV Discharge Air Temperature (°K),"East Zone VAV Damper Position (fraction, 0: fully closed to 1: fully open)","East Zone VAV Damper Control Signal (fraction, 0: damper should be fully closed to 1: damper should be fully open)",East Zone VAV Discharge Air Flow Rate (m3/s),...,Power Consumption of the Chilled Water Pump (W),Power Consumption of the Cooling Water Pump (W),Power Consumption of the Hot Water Pump (W),Power Consumption of the Cooling Tower Fan (W),"Speed Control Signal of the Chilled Water Pump (fraction, 0: pump speed should be 0% to 1: pump speed should be 100%)","Speed Control Signal of the Cooling Water Pump (fraction, 0: pump speed should be 0% to 1: pump speed should be 100%)","Speed Control Signal of the Hot Water Pump (fraction, 0: pump speed should be 0% to 1: pump speed should be 100%)","Cooling Mode Control Signal (1: free cooling , 2: partial mechanical, 3: full mechanical, 4: off)","Chiller Plant Equipment Operation Signal (0: off, 1: on)","Boiler Plant Equipment Operation Signal (0: off, 1: on)"
0,17625600,297.149994,303.149994,285.149994,0.0,-0.0,293.149994,0.999000,0.0,2.558512e-13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
1,17625660,296.997681,303.149994,285.149994,0.0,-0.0,293.149994,0.343101,0.0,2.006035e-03,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
2,17625720,296.894867,303.149994,285.149994,0.0,-0.0,293.211884,0.000000,0.0,4.749701e-04,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
3,17625780,296.836914,303.149994,285.149994,0.0,-0.0,293.211975,0.000000,0.0,1.106418e-04,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
4,17625840,296.810669,303.149994,285.149994,0.0,-0.0,293.212036,0.000000,0.0,1.034468e-04,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0



Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/OADamStuck_65.csv
Rows: 10,081; fields: 114


,pandas_dtype
time (s),int64
East Zone Room Temperature (°K),float64
East Zone Room Cooling Setpoint (°K),float64
East Zone Room Heating Setpoint (°K),float64
East Zone Reheating Coil Valve Power Control Signal (W),float64
...,...
"Speed Control Signal of the Cooling Water Pump (fraction, 0: pump speed should be 0% to 1: pump speed should be 100%)",float64
"Speed Control Signal of the Hot Water Pump (fraction, 0: pump speed should be 0% to 1: pump speed should be 100%)",float64
"Cooling Mode Control Signal (1: free cooling , 2: partial mechanical, 3: full mechanical, 4: off)",float64
"Chiller Plant Equipment Operation Signal (0: off, 1: on)",float64


,time (s),East Zone Room Temperature (°K),East Zone Room Cooling Setpoint (°K),East Zone Room Heating Setpoint (°K),East Zone Reheating Coil Valve Power Control Signal (W),East Zone Reheating Coil Valve Power Consumption (W),East Zone VAV Discharge Air Temperature (°K),"East Zone VAV Damper Position (fraction, 0: fully closed to 1: fully open)","East Zone VAV Damper Control Signal (fraction, 0: damper should be fully closed to 1: damper should be fully open)",East Zone VAV Discharge Air Flow Rate (m3/s),...,Power Consumption of the Chilled Water Pump (W),Power Consumption of the Cooling Water Pump (W),Power Consumption of the Hot Water Pump (W),Power Consumption of the Cooling Tower Fan (W),"Speed Control Signal of the Chilled Water Pump (fraction, 0: pump speed should be 0% to 1: pump speed should be 100%)","Speed Control Signal of the Cooling Water Pump (fraction, 0: pump speed should be 0% to 1: pump speed should be 100%)","Speed Control Signal of the Hot Water Pump (fraction, 0: pump speed should be 0% to 1: pump speed should be 100%)","Cooling Mode Control Signal (1: free cooling , 2: partial mechanical, 3: full mechanical, 4: off)","Chiller Plant Equipment Operation Signal (0: off, 1: on)","Boiler Plant Equipment Operation Signal (0: off, 1: on)"
0,17625600,297.149994,303.149994,285.149994,0.0,-0.0,293.149994,0.999,0.0,2.558512e-13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
1,17625660,296.894867,303.149994,285.149994,0.0,-0.0,293.158081,0.000,0.0,4.823937e-04,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
2,17625720,296.894867,303.149994,285.149994,0.0,-0.0,293.158081,0.000,0.0,4.823937e-04,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
3,17625780,296.810364,303.149994,285.149994,0.0,-0.0,293.158081,0.000,0.0,1.028440e-04,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
4,17625840,296.810364,303.149994,285.149994,0.0,-0.0,293.158081,0.000,0.0,1.028440e-04,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0


In [27]:
# First quality checks: missing cells, exact repeated rows, and
# unexpected numeric ranges. These checks do not establish ground truth.
quality_rows = []
for name, frame in frames.items():
    missing_by_column = frame.isna().sum().sort_values(ascending=False)
    numeric = frame.select_dtypes(include="number")
    quality_rows.append({
        "csv": name,
        "rows": len(frame),
        "fields": len(frame.columns),
        "missing_cells": int(frame.isna().sum().sum()),
        "columns_with_missing": int((missing_by_column > 0).sum()),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
    })
    print(f"\n{name} — columns with missing entries (up to 15):")
    display(missing_by_column[missing_by_column > 0].head(15).to_frame("missing"))
    print("Numeric ranges (up to 20 fields; units not assumed):")
    display(numeric.agg(["min", "max"]).T.head(20))

quality_summary = pd.DataFrame(quality_rows)
display(quality_summary)
if len(frames) > 1:
    common = set.intersection(*(set(df.columns) for df in frames.values()))
    union = set.union(*(set(df.columns) for df in frames.values()))
    print(f"Field overlap across inspected runs: {len(common)} common of {len(union)} distinct.")
    print("This is a selected-run inventory, not a source/target channel or unit audit.")
print("Scenario types cannot be verified from two file previews. Consult 00_explanations.pdf.")



Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/BaselineSystem.csv — columns with missing entries (up to 15):


,missing


Numeric ranges (up to 20 fields; units not assumed):


,min,max
time (s),1.762560e+07,1.823040e+07
East Zone Room Temperature (°K),2.968033e+02,2.991206e+02
East Zone Room Cooling Setpoint (°K),2.971500e+02,3.031500e+02
East Zone Room Heating Setpoint (°K),2.851500e+02,2.931500e+02
East Zone Reheating Coil Valve Power Control Signal (W),0.000000e+00,0.000000e+00
East Zone Reheating Coil Valve Power Consumption (W),-0.000000e+00,-0.000000e+00
East Zone VAV Discharge Air Temperature (°K),2.845961e+02,2.936377e+02
"East Zone VAV Damper Position (fraction, 0: fully closed to 1: fully open)",0.000000e+00,9.990079e-01
"East Zone VAV Damper Control Signal (fraction, 0: damper should be fully closed to 1: damper should be fully open)",0.000000e+00,7.155870e-01
East Zone VAV Discharge Air Flow Rate (m3/s),-1.629354e-08,6.103312e-01



Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/OADamStuck_65.csv — columns with missing entries (up to 15):


,missing


Numeric ranges (up to 20 fields; units not assumed):


,min,max
time (s),1.762560e+07,1.823040e+07
East Zone Room Temperature (°K),2.968034e+02,2.991187e+02
East Zone Room Cooling Setpoint (°K),2.971500e+02,3.031500e+02
East Zone Room Heating Setpoint (°K),2.851500e+02,2.931500e+02
East Zone Reheating Coil Valve Power Control Signal (W),0.000000e+00,0.000000e+00
East Zone Reheating Coil Valve Power Consumption (W),-0.000000e+00,-0.000000e+00
East Zone VAV Discharge Air Temperature (°K),2.846911e+02,2.936303e+02
"East Zone VAV Damper Position (fraction, 0: fully closed to 1: fully open)",0.000000e+00,9.990000e-01
"East Zone VAV Damper Control Signal (fraction, 0: damper should be fully closed to 1: damper should be fully open)",0.000000e+00,7.312813e-01
East Zone VAV Discharge Air Flow Rate (m3/s),-4.182891e-05,6.183872e-01


,csv,rows,fields,missing_cells,columns_with_missing,exact_duplicate_rows
0,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...,10081,114,0,0,0
1,Public_ScientificData_AHUFaults/04_G36-1wk/Coo...,10081,114,0,0,0


Field overlap across inspected runs: 114 common of 114 distinct.
This is a selected-run inventory, not a source/target channel or unit audit.
Scenario types cannot be verified from two file previews. Consult 00_explanations.pdf.


In [28]:
# Compare only *raw field names* as a first warning about compatibility.
# Identical names do not prove identical physical units or operating modes.
for source_name, source_frame in frames.items():
    for target_name, target_frame in target_frames.items():
        common_fields = sorted(set(source_frame.columns) & set(target_frame.columns))
        print(f"{source_name} vs {target_name}: {len(common_fields)} shared raw column names")
        print("First 20 shared names:", common_fields[:20])
print("A manual channel/unit/season mapping is needed before source-to-target modeling.")


Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/BaselineSystem.csv vs SZVAV.csv: 0 shared raw column names
First 20 shared names: []
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/BaselineSystem.csv vs SZCAV.csv: 0 shared raw column names
First 20 shared names: []
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/OADamStuck_65.csv vs SZVAV.csv: 0 shared raw column names
First 20 shared names: []
Public_ScientificData_AHUFaults/04_G36-1wk/CoolingSeason/OADamStuck_65.csv vs SZCAV.csv: 0 shared raw column names
First 20 shared names: []
A manual channel/unit/season mapping is needed before source-to-target modeling.


## Interpretation and open issues

- Treat the tables above as observations for the inspected files only. Use their actual numbers in the Canvas note; do not infer complete-source class balance from two runs.
- The source is simulated and the target is from real FLEXLAB operation. Differences in sensors, units, operating mode, season and simulation artifacts remain to be audited before model training.
- Both source and target licenses are documented above, but the complete source class counts and exact cross-setting channel mapping have **not** been established by this warm-up. Keep scenario/run IDs intact and split at the run level before making time windows.
- After running, write one specific numeric finding from `quality_summary` into the submission note. If a selected path is the wrong scenario, set `DATA_CSV_NAMES` to exact paths from the inventory and rerun.
